# 01 — Calidad de datos y calibración

Cartera sintética de la **Compañía de Seguros Cordillera (ficticia)**: seguros colectivos de salud complementaria y vida. Datos al corte de valoración **31-12-2025** (solo siniestros pagados hasta esa fecha).

Este notebook revisa shape, tipos y nulos de cada tabla de staging, la composición de la cartera, frecuencia y severidad por tipo de prestación, la siniestralidad anual y la calibración contra objetivos de mercado.

Requiere haber corrido el pipeline (`data/warehouse.duckdb`).

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

from colectivos.calibration import calibration_report

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / 'data' / 'warehouse.duckdb'
con = duckdb.connect(str(DB_PATH), read_only=True)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 160)

## 1. Shape, tipos y nulos por tabla de staging

In [2]:
tables = [row[0] for row in con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'staging' ORDER BY 1"
).fetchall()]
summary = []
for table in tables:
    frame = con.execute(f'SELECT * FROM staging.{table}').df()
    summary.append({'tabla': table, 'filas': frame.shape[0], 'columnas': frame.shape[1],
                    'nulos': int(frame.isna().sum().sum())})
pd.DataFrame(summary)

,tabla,filas,columnas,nulos
0,stg_companies,300,5,245
1,stg_contracts,1704,11,544
2,stg_exposure_monthly,3107310,5,0
3,stg_health_claims,3127698,10,0
4,stg_insureds,91418,9,107946
5,stg_life_claims,399,5,0
6,stg_plans,15,5,0
7,stg_premiums_monthly,14796,3,0


Los únicos nulos esperados son `lapse_date` (empresas que siguen vigentes), `holder_id` (titulares), `termination_date` (asegurados vigentes) y `rate_change_applied` (primera vigencia y vigencia 2026).

In [3]:
null_detail = []
for table in tables:
    frame = con.execute(f'SELECT * FROM staging.{table}').df()
    nulls = frame.isna().sum()
    null_detail += [{'tabla': table, 'columna': col, 'nulos': int(n)} for col, n in nulls.items() if n > 0]
pd.DataFrame(null_detail)

,tabla,columna,nulos
0,stg_companies,lapse_date,245
1,stg_contracts,rate_change_applied,544
2,stg_insureds,holder_id,53759
3,stg_insureds,termination_date,54187


In [4]:
con.execute('SELECT * FROM staging.stg_health_claims').df().dtypes

claim_id                        str
insured_id                      str
company_id                      str
contract_id                     str
claim_type                      str
occurrence_date      datetime64[us]
payment_date         datetime64[us]
billed_uf                   float64
system_covered_uf           float64
reimbursed_uf               float64
dtype: object

## 2. Composición de la cartera

In [5]:
con.execute('''
    SELECT age_band, role, ROUND(SUM(exposure) / 12, 0) AS anios_persona
    FROM intermediate.int_insured_profile_month
    GROUP BY ALL ORDER BY age_band, role
''').df().pivot(index='age_band', columns='role', values='anios_persona')

role,carga,titular
age_band,,
0-17,"38,848.0000",NaN
18-29,"29,413.0000","17,057.0000"
30-44,"19,097.0000","65,870.0000"
45-59,"15,601.0000","57,028.0000"
60-69,"2,595.0000","9,989.0000"
70+,378.0000,999.0000


In [6]:
con.execute('''
    SELECT year, health_system, SUM(exposure) / 12 AS anios_persona
    FROM intermediate.int_insured_profile_month
    GROUP BY ALL
''').df().pivot(index='year', columns='health_system', values='anios_persona').pipe(
    lambda d: d.div(d.sum(axis=1), axis=0)).rename(columns=lambda c: f'share_{c}')

health_system,share_fonasa,share_isapre
year,,
2021,0.2004,0.7996
2022,0.2019,0.7981
2023,0.2026,0.7974
2024,0.2198,0.7802
2025,0.2380,0.7620


La proporción Fonasa sube desde 2024: es el patrón plantado `fonasa_shift`.

## 3. Frecuencia y severidad por tipo de prestación

In [7]:
freq_sev = con.execute('''
    SELECT year, claim_type,
           SUM(n_claims) / SUM(exposure_years) AS frecuencia,
           SUM(reimbursed_uf) / NULLIF(SUM(n_claims), 0) AS severidad_uf
    FROM marts.mart_model_cells
    GROUP BY ALL ORDER BY ALL
''').df()
freq_sev.pivot(index='claim_type', columns='year', values='frecuencia')

year,2021,2022,2023,2024,2025
claim_type,,,,,
alto_costo,0.0047,0.0048,0.0048,0.0047,0.0027
ambulatoria,6.1088,6.6754,6.9562,7.0960,6.6939
dental,0.7238,0.7572,0.7736,0.7758,0.6951
farmacia,4.2258,4.5264,4.7048,4.7709,4.7012
hospitalaria,0.1325,0.1371,0.1468,0.1480,0.1153


In [8]:
freq_sev.pivot(index='claim_type', columns='year', values='severidad_uf')

year,2021,2022,2023,2024,2025
claim_type,,,,,
alto_costo,140.8371,174.0890,214.5123,194.9753,207.7842
ambulatoria,0.4088,0.4292,0.4510,0.4799,0.5058
dental,1.4626,1.5139,1.5688,1.6488,1.7000
farmacia,0.3558,0.3744,0.3954,0.4147,0.4368
hospitalaria,17.5041,19.3084,19.9968,21.4949,24.4652


2025 aparece más bajo en frecuencia, sobre todo en hospitalaria y alto costo: son siniestros ocurridos que aún no se pagaban al corte (IBNR). La fase 2 estima ese monto con chain-ladder.

## 4. Siniestralidad anual

In [9]:
con.execute('''
    SELECT year,
           SUM(earned_premium_uf) AS prima_uf,
           SUM(health_claims_uf) AS salud_uf,
           SUM(life_claims_uf) AS vida_uf,
           SUM(health_claims_uf + life_claims_uf) / SUM(earned_premium_uf) AS siniestralidad
    FROM marts.mart_loss_ratio_contract_year
    GROUP BY year ORDER BY year
''').df()

,year,prima_uf,salud_uf,vida_uf,siniestralidad
0,2021,"580,291.4451","414,514.9585","22,400.0000",0.7529
1,2022,"582,296.4400","458,287.3595","19,800.0000",0.8210
2,2023,"590,336.8516","489,652.7191","21,100.0000",0.8652
3,2024,"672,779.9625","568,899.1669","22,100.0000",0.8784
4,2025,"720,152.0434","544,503.3252","21,100.0000",0.7854


## 5. Calibración

Rangos objetivo y fuentes en `config/calibration_targets.yaml`. Todos son supuestos razonados: no se encontró una cifra oficial publicada para colectivos de salud.

In [10]:
calibration_report(DB_PATH, PROJECT_ROOT / 'config' / 'calibration_targets.yaml')[['metric', 'generated', 'target_low', 'target_high', 'assumption', 'within_range']]

,metric,generated,target_low,target_high,assumption,within_range
0,loss_ratio_health_annual,0.7962,0.7500,0.9500,True,True
1,claims_cost_per_insured_uf,9.5414,6.0000,12.0000,True,True
2,health_severity_trend,0.0540,0.0400,0.0900,True,True
3,fonasa_share_start,0.2004,0.1500,0.3000,True,True
